# 🎯 5. Planning and Reasoning Patterns

Reasoning is what makes an agent *intelligent*. The LLM's ability to think step-by-step, decompose problems, and evaluate its own plans is the core differentiator from simple automation.

In this notebook:

1. **Chain-of-Thought (CoT)** — step-by-step reasoning
2. **Task decomposition** — breaking complex problems into sub-tasks
3. **Planner-Executor pattern** — separate planning from execution
4. **Tree of Thought (ToT)** — exploring multiple reasoning paths
5. **Graph of Thought (GoT)** — combining multiple perspectives
6. **Self-reflection** — agents that evaluate their own output
7. **Reasoning topology comparison** — when to use each pattern

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

LLM_MODEL   = "gpt-4o-mini"

client = OpenAI()
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
print(f'Model: {LLM_MODEL}')

Model: gpt-4o-mini


## 5.1 Chain-of-Thought (CoT) Reasoning

**Chain-of-Thought** prompting makes the LLM show its reasoning step by step. This significantly improves accuracy on complex tasks.

```
Without CoT: "The answer is 42"
With CoT:    "First, I need to... Then... Therefore... The answer is 42"
```

For agents, CoT is built into the **Thought** step of the agent loop.

In [5]:
# Compare: without vs with Chain-of-Thought
from IPython.display import display, Markdown

problem = (
    'A store has 3 shelves. The first shelf has twice as many books as the second. '
    'The third shelf has 5 fewer books than the first. '
    'If the total is 55 books, how many books are on each shelf?'
)

# Without CoT
r1 = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': f'Answer directly: {problem} (NO LaTeX)'}],
    temperature=0
)
display(Markdown('\n ### ------------WITHOUT CoT------------'))
display(Markdown(r1.choices[0].message.content))

# With CoT
r2 = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': f'Think step by step and show your work: {problem} (NO LaTeX)'}],
    temperature=0
)
display(Markdown('\n ### ------------WITH CoT------------'))
display(Markdown(r2.choices[0].message.content))


 ### ------------WITHOUT CoT------------

Let the number of books on the second shelf be x. Then, the first shelf has 2x books, and the third shelf has (2x - 5) books. 

The equation for the total number of books is:
x + 2x + (2x - 5) = 55.

Combining like terms gives:
5x - 5 = 55.

Adding 5 to both sides:
5x = 60.

Dividing by 5:
x = 12.

Now, we can find the number of books on each shelf:
- Second shelf: 12 books.
- First shelf: 2x = 2(12) = 24 books.
- Third shelf: 2x - 5 = 24 - 5 = 19 books.

So, the number of books on each shelf is:
First shelf: 24 books,
Second shelf: 12 books,
Third shelf: 19 books.


 ### ------------WITH CoT------------

Let's define the number of books on each shelf using variables:

1. Let the number of books on the second shelf be x.
2. Then, the number of books on the first shelf, which has twice as many books as the second shelf, will be 2x.
3. The number of books on the third shelf, which has 5 fewer books than the first shelf, will be 2x - 5.

Now, we can set up an equation based on the total number of books:

Total books = Books on first shelf + Books on second shelf + Books on third shelf

This can be written as:

2x + x + (2x - 5) = 55

Now, let's simplify the left side of the equation:

2x + x + 2x - 5 = 55
5x - 5 = 55

Next, we will add 5 to both sides of the equation:

5x - 5 + 5 = 55 + 5
5x = 60

Now, we will divide both sides by 5 to solve for x:

5x / 5 = 60 / 5
x = 12

Now that we have the value of x, we can find the number of books on each shelf:

1. Books on the second shelf (x) = 12
2. Books on the first shelf (2x) = 2 * 12 = 24
3. Books on the third shelf (2x - 5) = 24 - 5 = 19

So, the number of books on each shelf is:

- First shelf: 24 books
- Second shelf: 12 books
- Third shelf: 19 books

To verify, we can check the total:

24 + 12 + 19 = 55

The calculations are correct. Therefore, the final answer is:

- First shelf: 24 books
- Second shelf: 12 books
- Third shelf: 19 books

## 5.2 Task Decomposition — The Planner-Executor Pattern

Complex tasks should be broken into smaller, manageable sub-tasks. The **Planner-Executor** pattern separates:

- **Planner**: LLM creates a structured plan
- **Executor**: Code executes each step

This is safer than letting the agent improvise — the plan can be reviewed before execution.

In [7]:
from pydantic import BaseModel, Field
from typing import List, Literal

class SubTask(BaseModel):
    """A single step in the execution plan"""
    step_number: int = Field(description="Order of execution")
    description: str = Field(description="What this step does")
    tool_needed: str = Field(description="Tool to use: search, calculate, write or none")
    depends_on: List[int] = Field(default=[], description="Step numbers this depends on")


class ExecutionPlan(BaseModel):
    """A structured plan for completing a complex task"""
    goal: str = Field(description="The main goal")
    subtasks: List[SubTask] = Field(description="Ordered list of subtasks")
    estimated_steps: int = Field(description="Total number of steps")

planner = llm.with_structured_output(ExecutionPlan)

plan = planner.invoke(
    'Create a plan to: Research the top 3 AI Agent Frameworks'
    "compare their features, and write a recommendation report"
)

print(f"Goal: {plan.goal}")
print(f"Steps: {plan.estimated_steps}\n")

for task in plan.subtasks:
    deps = f" depends on: {task.depends_on}" if task.depends_on else ''
    print(f"  {task.step_number}. [{task.tool_needed}] {task.description}{deps}")



Goal: Research the top 3 AI Agent Frameworks, compare their features, and write a recommendation report.
Steps: 5

  1. [search] Identify the top 3 AI Agent Frameworks based on popularity and usage.
  2. [search] Gather detailed information about the features of each of the identified frameworks. depends on: [1]
  3. [none] Analyze the gathered information to compare the features of the frameworks. depends on: [2]
  4. [write] Draft a recommendation report based on the comparison analysis. depends on: [3]
  5. [write] Review and finalize the recommendation report for clarity and completeness. depends on: [4]


In [8]:
def execute_plan(plan: ExecutionPlan) -> dict:
    """Execute a plan step by step, collection results"""
    results = {}

    for task in plan.subtasks:
        display(Markdown(f"\n\n#### **Executing Step {task.step_number}**: {task.description}"))

        # Check dependencies
        dep_context = ''
        for dep in task.depends_on:
            if dep in results:
                dep_context += f"\nResult from step {dep}: {results[dep]}"
        
        # Execute based on loop type
        if task.tool_needed == "none":
            result = f"**Step {task.step_number}** completed (no tool needed)"
        else:
            # In production this would call real tools !!!
            prompt = f"#### Execute this task: {task.description}"
            if dep_context:
                prompt += f"\n\nContext from previous steps: {dep_context}"

            response = llm.invoke([
                SystemMessage(content="Complete the task concisely. Use the content provided"),
                HumanMessage(content=prompt)
            ]) 
            result = response.content
        
        results[task.step_number] = result
        display(Markdown(f"#### Result: {result}"))
    
    return results

results = execute_plan(plan)
display(Markdown(f"\n\n"))
display(Markdown(f"### Plan completed! {len(results)} steps executed"))



#### **Executing Step 1**: Identify the top 3 AI Agent Frameworks based on popularity and usage.

#### Result: The top 3 AI Agent Frameworks based on popularity and usage are:

1. **Rasa** - An open-source framework for building conversational AI and chatbots.
2. **Microsoft Bot Framework** - A comprehensive framework for developing bots that can interact across multiple channels.
3. **OpenAI Gym** - A toolkit for developing and comparing reinforcement learning algorithms, widely used in AI research.



#### **Executing Step 2**: Gather detailed information about the features of each of the identified frameworks.

#### Result: Here are the detailed features of each of the identified AI Agent Frameworks:

### 1. Rasa
- **Open Source**: Rasa is completely open-source, allowing developers to customize and extend the framework as needed.
- **Natural Language Understanding (NLU)**: Rasa provides robust NLU capabilities, enabling the extraction of intents and entities from user inputs.
- **Dialogue Management**: It uses a flexible dialogue management system that allows for complex conversation flows and context handling.
- **Custom Actions**: Developers can create custom actions to integrate with external APIs or databases, enhancing the bot's functionality.
- **Multi-language Support**: Rasa supports multiple languages, making it suitable for global applications.
- **Training and Evaluation**: It includes tools for training models and evaluating their performance, ensuring continuous improvement.
- **Integration**: Rasa can be integrated with various messaging platforms (e.g., Slack, Facebook Messenger) and can also be deployed on-premises or in the cloud.

### 2. Microsoft Bot Framework
- **Comprehensive SDK**: The framework provides a rich SDK that supports multiple programming languages, including C#, JavaScript, and Python.
- **Channel Integration**: It allows bots to be deployed across various channels such as Microsoft Teams, Skype, Slack, and more, ensuring wide reach.
- **Bot Framework Composer**: A visual authoring tool that simplifies bot development, enabling users to design conversational experiences without extensive coding.
- **Azure Integration**: Seamless integration with Azure services, including Azure Bot Service, for hosting and scaling bots.
- **Adaptive Cards**: Supports rich interactive content through adaptive cards, enhancing user engagement.
- **State Management**: Built-in state management capabilities to maintain conversation context and user data.
- **Security and Compliance**: Offers enterprise-grade security features, including authentication and compliance with various standards.

### 3. OpenAI Gym
- **Reinforcement Learning Environment**: Provides a variety of environments for testing and developing reinforcement learning algorithms, including classic control tasks and Atari games.
- **Standardized API**: Features a consistent API for environments, making it easier to switch between different tasks and compare algorithms.
- **Extensive Documentation**: Comprehensive documentation and tutorials to help users understand how to implement and evaluate reinforcement learning algorithms.
- **Community Contributions**: A large community that contributes additional environments and tools, expanding the framework's capabilities.
- **Integration with Libraries**: Compatible with popular machine learning libraries like TensorFlow and PyTorch, facilitating the development of complex models.
- **Benchmarking**: Tools for benchmarking algorithms against standard environments, allowing researchers to compare performance effectively.
- **Custom Environment Creation**: Users can create custom environments tailored to specific research needs or applications.

These features make each framework suitable for different applications in the field of AI, from conversational agents to reinforcement learning research.



#### **Executing Step 3**: Analyze the gathered information to compare the features of the frameworks.

...



#### **Executing Step 4**: Draft a recommendation report based on the comparison analysis.

#### Result: ### Recommendation Report

**Subject:** Comparison Analysis Results and Recommendations

**Date:** [Insert Date]

**Prepared by:** [Your Name]

---

#### Executive Summary

This report presents the findings from the recent comparison analysis conducted on [insert subject of analysis]. The analysis evaluated [insert key criteria or metrics] across [insert number of options or subjects compared]. Based on the results, this report provides recommendations for the best course of action.

#### Analysis Overview

The comparison analysis focused on the following key areas:

1. **Criteria 1:** [Brief description of findings]
2. **Criteria 2:** [Brief description of findings]
3. **Criteria 3:** [Brief description of findings]

Each option was assessed based on these criteria, leading to a comprehensive understanding of their strengths and weaknesses.

#### Findings

- **Option A:** [Summary of performance, strengths, and weaknesses]
- **Option B:** [Summary of performance, strengths, and weaknesses]
- **Option C:** [Summary of performance, strengths, and weaknesses]

#### Recommendations

Based on the analysis, the following recommendations are made:

1. **Recommendation 1:** [Detail the recommended option and rationale]
2. **Recommendation 2:** [If applicable, detail a secondary option or alternative]
3. **Recommendation 3:** [Any additional suggestions for implementation or further analysis]

#### Conclusion

The comparison analysis has provided valuable insights into the options available. Implementing the recommended actions will [insert expected outcomes or benefits]. Further monitoring and evaluation should be conducted to ensure the effectiveness of the chosen option.

---

**Appendices:** [Include any additional data or charts that support the analysis]

**Contact Information:** [Your contact details for follow-up questions] 

--- 

**End of Report**



#### **Executing Step 5**: Review and finalize the recommendation report for clarity and completeness.

#### Result: ### Recommendation Report

**Subject:** Comparison Analysis Results and Recommendations

**Date:** [Insert Date]

**Prepared by:** [Your Name]

---

#### Executive Summary

This report presents the findings from the recent comparison analysis conducted on [insert subject of analysis]. The analysis evaluated [insert key criteria or metrics] across [insert number of options or subjects compared]. Based on the results, this report provides recommendations for the best course of action.

#### Analysis Overview

The comparison analysis focused on the following key areas:

1. **Criteria 1:** [Brief description of findings]
2. **Criteria 2:** [Brief description of findings]
3. **Criteria 3:** [Brief description of findings]

Each option was assessed based on these criteria, leading to a comprehensive understanding of their strengths and weaknesses.

#### Findings

- **Option A:** [Summary of performance, strengths, and weaknesses]
- **Option B:** [Summary of performance, strengths, and weaknesses]
- **Option C:** [Summary of performance, strengths, and weaknesses]

#### Recommendations

Based on the analysis, the following recommendations are made:

1. **Recommendation 1:** [Detail the recommended option and rationale]
2. **Recommendation 2:** [If applicable, detail a secondary option or alternative]
3. **Recommendation 3:** [Any additional suggestions for implementation or further analysis]

#### Conclusion

The comparison analysis has provided valuable insights into the options available. Implementing the recommended actions will [insert expected outcomes or benefits]. Further monitoring and evaluation should be conducted to ensure the effectiveness of the chosen option.

---

**Appendices:** [Include any additional data or charts that support the analysis]

**Contact Information:** [Your contact details for follow-up questions] 

--- 

**End of Report**

---

### Final Review Notes:
- Ensure all placeholders are filled with relevant information.
- Verify that the descriptions and summaries are concise and clear.
- Check for consistency in formatting and terminology throughout the report.
- Confirm that all recommendations are actionable and supported by the findings.

### Plan completed! 5 steps executed

## 5.3 Implementing the Planner-Executor Pattern

## 5.4 Tree of Thought (ToT) — Exploring Multiple Paths

Instead of a single chain of reasoning, **Tree of Thought** explores multiple reasoning branches and selects the best:

<img src="images/tree-of-thought.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

**Key operations**: Branch → Score → Prune → Select best

## 5.5 Self-Reflection — Agents That Check Their Own Work

Self-reflection is a powerful pattern where the agent **evaluates its own output** and iterates if needed:

<img src="images/self-refrection.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

## 5.6 Reasoning Topology Comparison

| Topology | Shape | Key Operation | Best For | LLM Calls |
|----------|-------|--------------|---------|----------|
| **CoT** | Linear chain | Step-by-step reasoning | Simple multi-step problems | 1 |
| **ToT** | Tree (beam search) | Branch → Score → Prune | Competitive path selection | 3-5+ |
| **GoT** | DAG (graph) | Branch → Contrast → Merge | Multi-perspective synthesis | 4-6+ |
| **AoT** | Planned DAG | Plan → Validate → Execute | Auditable production pipelines | 3+ |
| **ReAct** | Loop | Think → Act → Observe | Dynamic exploration with tools | Variable |
| **Self-Reflect** | Loop | Generate → Evaluate → Improve | Quality-critical tasks | 2-6 |

### When to Use Each

- **CoT**: Default for most tasks — simple and effective
- **ToT**: When you need to explore alternatives and pick the best
- **GoT**: When multiple perspectives need to be synthesized
- **AoT**: When execution must be deterministic and auditable
- **ReAct**: When the agent needs to interact with tools dynamically
- **Self-Reflect**: When output quality is critical and iteration is affordable

## 💡 Exercise 5: Build a Planner-Executor Agent

**Task**: Build a planner-executor system that:
1. Takes a research question
2. Creates a structured plan (Pydantic model)
3. Executes each step
4. Self-evaluates the final output

Test with: *"What are the key differences between LangGraph and CrewAI for building multi-agent systems?"*

In [ ]:
# Exercise 5: YOUR CODE HERE


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **CoT** | "Think step by step" dramatically improves LLM reasoning |
| **Task Decomposition** | Break complex tasks into structured sub-tasks with dependencies |
| **Planner-Executor** | Separate planning (LLM) from execution (code) for safety |
| **Tree of Thought** | Explore multiple paths, score and select the best |
| **Self-Reflection** | Generate → Evaluate → Improve loop for quality-critical output |

### What's Next

In **Notebook 06: Memory in AI Agents**, we give our agents the ability to remember — short-term conversation memory, long-term persistence, and context engineering strategies.